# Text-Based Depression Detection — Baseline Classification Report

**Transcription / Text Branch**

| | |
|---|---|
| **Prepared by** | Parth Shringarpure |
| **Supervisor** | Prof. Alessandro Vinciarelli |
| **Project** | Depression Detection from Spontaneous Speech (Androids Corpus) |
| **Primary reference** | Alsarrani, Esposito & Vinciarelli (ICMI 2025), *Punctual or Continuous?* |
| **Scope** | Text branch only (audio/speech branch handled separately) |
| **Baseline configuration** | Single Instance Learning (SIL), FFNN classifier, one training run per fold |
| **Date** | **[Information Missing – Please Complete: date of report]** |
| **Document status** | Reference document — baseline established prior to RNN and MIL experiments |

---

## Table of Contents

1. Executive Summary
2. Baseline Overview
3. Alessandro's Paper (Baseline Reference)
4. Dataset
5. Experimental Setup
6. Baseline Implementation
7. Evaluation
8. Results Summary
9. Discussion
10. Foundation for Future Work
11. References

---

## 1. Executive Summary

### 1.1 Purpose of the baseline

This report documents the first complete, end-to-end implementation of the **text (transcription) branch** of a depression-detection system built on the Androids Corpus. It transforms Italian spontaneous-speech recordings into transcripts, segments those transcripts, converts each segment into a numerical feature representation, and classifies each speaker as belonging to either the **Control (HC)** or **Depression (PT)** class. The baseline uses a Feed-Forward Neural Network (FFNN) under the Single Instance Learning (SIL) paradigm, evaluated with the corpus's own 5-fold, person-independent cross-validation protocol.

### 1.2 Why establishing a baseline matters

A baseline provides a fixed, well-documented reference point against which all subsequent work in the dissertation can be measured. Specifically, this baseline:

- **Anchors comparison.** Later architectures (a recurrent neural network; a Multiple Instance Learning variant) can only be judged as "better" or "worse" relative to a known, reproducible starting point.
- **Validates the pipeline.** It confirms that every upstream stage — transcription, segmentation, feature extraction, fold assignment — is correct and free of data leakage before more complex modelling is attempted.
- **Establishes methodological alignment with the reference paper.** By matching the classifier architecture and evaluation protocol of the reference paper as closely as possible, the baseline makes the eventual results directly comparable to published benchmarks.
- **Documents a key deviation for the supervisor.** Because institutional access to the reference paper's feature-extraction tool (LIWC) was unconfirmed at the time of this work, an alternative feature representation was used. The baseline documents this substitution explicitly so its impact can be discussed and, if necessary, revisited.

### 1.3 Headline result

Using text alone, the best-performing configuration (N = 1) achieved a recording-level **F1 score of 90.27% (± 3.95%)** and **accuracy of 88.80% (± 5.87%)**. This exceeds the reference paper's best *text-only* published result at every segmentation level tested and approaches the paper's best *multimodal* (speech + text) result. These figures should be read alongside the limitations in Sections 9 and the fact that they derive from a single training run per fold (Section 5).

---

## 2. Baseline Overview

### 2.1 Baseline approach

The baseline is a supervised binary classification pipeline operating entirely on the textual transcriptions of interview recordings. It follows the sequence:

> Audio → Transcript → Segments → Feature vectors → Per-segment classification → Per-speaker decision

The classifier is a Feed-Forward Neural Network trained under the **Single Instance Learning (SIL)** paradigm: each text segment is classified independently, and a speaker's final label is obtained by **majority vote** over all of that speaker's segment-level predictions.

### 2.2 Objectives

- Reproduce the text-branch methodology of the reference paper as faithfully as the available tools allow.
- Produce recording-level (per-speaker) results directly comparable to published benchmarks.
- Evaluate performance across all seven segmentation granularities (N = 1, 2, 4, 8, 16, 32, 64) to examine how classification behaves as segments become shorter.
- Verify correctness at every stage (counts, shapes, reconstruction, no speaker leakage).

### 2.3 Methodology (structured summary)

| Stage | Method | Output |
|---|---|---|
| 1. Transcription | Whisper `large-v3`, Italian forced | 116 transcripts |
| 2. Segmentation | Recursive binary split (word-count) into N = 1…64 | 14,732 segments |
| 3. Feature extraction | Mean-pooled Italian-BERT embeddings (768-dim) | 14,732 feature vectors |
| 4. Classification | FFNN (32-64-128, ReLU, sigmoid), SIL | Per-segment probability |
| 5. Aggregation | Majority vote per speaker | Per-speaker (recording-level) label |
| 6. Evaluation | 5-fold, person-independent cross-validation | Accuracy, F1 (mean ± std) |

### 2.4 Assumptions

- **All interview recordings are Italian**, justifying the forced-language transcription setting.
- **The folder label (HC / PT) is the ground-truth class label** for every segment belonging to that speaker.
- **"Equal parts" in the reference paper's segmentation refers to a defensible unit for text.** As the paper does not specify a unit for the text modality, word count was chosen (see Section 6.3).
- **All segments from one speaker share that speaker's single true label**, which is what makes recording-level majority voting well-defined.

### 2.5 Limitations (summary; expanded in Section 9)

- Feature extraction uses BERT embeddings, not the LIWC representation of the reference paper.
- Only the SIL paradigm is implemented; MIL is not.
- Only one training run per fold is performed (no repeated-initialisation averaging).
- No random seed is set, so exact numerical results are not bit-for-bit reproducible.

---

## 3. Alessandro's Paper (Baseline Reference)

The primary reference for this baseline is:

> **Alsarrani, R., Esposito, A., & Vinciarelli, A. (2025).** *Punctual or Continuous? Analyzing Depression Traces in Language and Paralanguage with Multiple Instance Learning.* Proceedings of ICMI 2025.

### 3.1 Original methodology in the paper

The paper investigates whether depression traces in speech and its transcriptions are **punctual** (detectable only at specific moments) or **continuous** (detectable at any moment). It processes both speech signals and their transcriptions in parallel through a common pipeline:

1. **Segmentation** (Section 3.1 of the paper): each recording and its transcription is recursively split into two equal parts, repeatedly, until N = 64 segments are produced.
2. **Feature extraction** (Section 3.2): for **transcriptions**, each segment is converted into a 94-dimensional vector using **LIWC** (Linguistic Inquiry and Word Count). For **speech**, handcrafted IS09 features (32-dim) and wav2vec2.0 embeddings (1024-dim) are used.
3. **Classification** (Sections 3.3–3.4): a Feed-Forward Neural Network with three hidden layers (32, 64, 128 neurons; ReLU) is trained under two paradigms — **Single Instance Learning (SIL)** and **Multiple Instance Learning (MIL)**.
4. **Aggregation** (Section 3.5): per-segment predictions are combined by **majority vote** to a per-recording decision (except for MIL on transcriptions, which classifies the bag directly).
5. **Evaluation**: 5-fold, person-independent cross-validation using the corpus's provided folds, each experiment repeated R = 5 times with different random initialisations; results reported as mean ± standard deviation.

### 3.2 Elements adopted directly from the paper (baseline)

The following are used **as-is** in this baseline and constitute the "reference" component:

| Adopted element | Source in paper |
|---|---|
| Segmentation method (recursive binary split to N = 64) | Section 3.1 |
| FFNN architecture (3 hidden layers: 32, 64, 128; ReLU; sigmoid output) | Sections 3.3–3.4 |
| Adam optimiser, binary cross-entropy loss, 30 epochs | Section 4 (experimental setup) |
| SIL classification paradigm | Section 3.3 |
| Majority-vote aggregation to recording level | Section 3.5 |
| 5-fold, person-independent cross-validation using the corpus's provided folds | Sections 3.3–3.4, 4.1 |
| Use of the Interview-Task (spontaneous speech) subset only | Section 4.1 |

### 3.3 Elements that are my own work (or adaptations)

The following clearly **depart from or extend** the paper and are **not** part of the paper's method:

| My work / adaptation | Relationship to paper |
|---|---|
| **Feature extraction with mean-pooled Italian-BERT embeddings (768-dim)** in place of LIWC (94-dim) | **Substitution.** Driven by unconfirmed LIWC access. Conceptually mirrors the paper's own handcrafted-vs-embeddings comparison on the speech side, applied here to text. |
| Choice of **word count** as the segmentation unit for text | **Clarification of an unspecified detail.** The paper does not state a unit for text segmentation. |
| Transcription with **Whisper large-v3, Italian forced** | The paper uses Whisper (per its Figure 1) but does not, to my reading, fix the exact model variant in the text; this is my concrete implementation choice. **[Information Missing – Please Complete: confirm exact Whisper variant used in paper, if stated]** |
| **PyTorch** implementation of the FFNN | **Adaptation.** The paper uses TensorFlow/Keras; the architecture and hyperparameters are matched, but the framework differs. |
| **Single run per fold** (R = 1) | **Simplification.** The paper uses R = 5 repeated initialisations; this baseline does not (yet). |
| **MIL not implemented** | **Omission (deliberate, for the baseline).** Only SIL is implemented so far. |

### 3.4 Secondary reference

> **Tao, F., Esposito, A., & Vinciarelli, A. (2023).** *The Androids Corpus: A New Publicly Available Benchmark for Speech Based Depression Detection.* Proceedings of Interspeech 2023.

This paper introduces the dataset and provides the corpus baseline benchmark (83.9% accuracy / 84.7% F1) used for comparison in Section 8.

---

## 4. Dataset

### 4.1 Description

The **Androids Corpus** (Tao, Esposito & Vinciarelli, 2023) is a publicly available benchmark for speech-based depression detection, consisting of spontaneous Italian speech recorded in mental-health settings. This work uses the **Interview-Task** subset only.

| Group | Meaning | Speakers | Class label |
|---|---|---|---|
| HC | Healthy Control | 52 | 0 (Control) |
| PT | Patient | 64 | 1 (Depression) |
| **Total** | | **116** | |

*Table 4.1 — Participant counts (Interview-Task).*

**Note on labelling convention:** the class folders are named `HC` and `PT`, while filenames encode the condition with a single letter — `C` (Control, inside `HC/`) or `P` (Patient, inside `PT/`). Filenames follow the pattern `nn_XGmm_t` (speaker id, condition letter, gender, age, education level).

### 4.2 Rationale for subset choice

Only the **Interview-Task** (spontaneous speech) is used — not the Reading-Task — for two reasons:

1. The reference paper reports experiments on spontaneous interview speech, so this keeps results comparable.
2. For the *text* branch specifically, Reading-Task transcripts are near-identical across all speakers (everyone reads the same passage), so they carry almost no speaker-discriminative linguistic signal.

The `audio_clip/` directory and `interview_timedata.csv` (which define a turn-level clip segmentation) were **not** used, because the reference paper's segmentation method operates on the whole recording, not on pre-cut conversational turns.

### 4.3 Preprocessing

| Step | Detail |
|---|---|
| Transcription | Whisper `large-v3`, `language="it"`, `task="transcribe"` |
| Text used | Plain transcript text (`.txt`); segment-level timestamps also saved (`.json`) but not used in this branch |
| Cleaning | Whitespace-based tokenisation only; no stopword removal, no lowercasing, no lemmatisation applied prior to embedding |

No additional filtering or record exclusion was performed; all 116 speakers are retained.

### 4.4 Segmentation and features

- **Segmentation:** each transcript is split into N = 1, 2, 4, 8, 16, 32, 64 word-count-equal contiguous segments, producing **14,732 segment files** in total (6,604 HC + 8,128 PT). Verified by exact word-for-word reconstruction (no data loss or duplication).
- **Features:** each segment is represented by a single **768-dimensional** mean-pooled embedding from `dbmdz/bert-base-italian-xxl-cased`. All 14,732 feature files verified for correct shape, absence of NaN values, absence of all-zero vectors, and non-identical outputs across speakers.

### 4.5 Splitting strategy

Evaluation uses the corpus's **pre-defined 5-fold cross-validation** split (`fold-lists.csv`), Interview-Task columns. The protocol is **person-independent**: all segments belonging to one speaker are assigned to exactly one fold, so no speaker ever appears in both training and test sets. This was **verified programmatically** — every one of the 116 speakers was confirmed to belong to exactly one fold, and the set of speakers parsed from feature filenames was confirmed to match the set of speakers in the fold list exactly.

Approximate fold sizes: one fold contains 24 speakers, the remaining four contain 23 speakers each (116 total).

---

## 5. Experimental Setup

### 5.1 Hardware

| Component | Detail |
|---|---|
| Transcription (Stage 1) | GPU (cloud); runs recorded on an NVIDIA Tesla T4 (Google Colab session). A later local CPU run is also present in the notebook history. |
| Feature extraction (Stage 3) | **[Information Missing – Please Complete: CPU or GPU used for embedding extraction]** |
| Classification (Stage 4) | **[Information Missing – Please Complete: confirm CPU/GPU; the classification notebook auto-selects CUDA if available, otherwise CPU]** |
| Local machine | Apple MacBook Air (Apple Silicon). **[Information Missing – Please Complete: exact model / chip / RAM]** |

### 5.2 Software and libraries

The classification pipeline uses the following libraries. **Exact installed versions on the machine where the notebook was run are not recorded in the notebook and must be completed manually:**

| Library | Purpose | Version |
|---|---|---|
| Python | Language runtime | **[Information Missing – Please Complete]** |
| PyTorch (`torch`) | FFNN definition, training, inference | **[Information Missing – Please Complete]** |
| Transformers (`transformers`) | Italian-BERT model + tokenizer (Stage 3) | **[Information Missing – Please Complete]** |
| scikit-learn (`sklearn`) | F1 score computation | **[Information Missing – Please Complete]** |
| NumPy (`numpy`) | Array handling, feature loading | **[Information Missing – Please Complete]** |
| pandas (`pandas`) | Reading `fold-lists.csv` | **[Information Missing – Please Complete]** |
| Whisper (`openai-whisper`) | Transcription (Stage 1) | **[Information Missing – Please Complete]** |
| tqdm | Progress bars | **[Information Missing – Please Complete]** |

### 5.3 Model, hyperparameters and configuration

| Parameter | Value | Source |
|---|---|---|
| Input dimension | 768 | Determined by BERT embedding size |
| Hidden layers | 3, sizes 32 → 64 → 128 | Reference paper |
| Activation | ReLU (hidden), Sigmoid (output) | Reference paper |
| Output | 1 unit (probability of Depression) | Reference paper |
| Optimiser | Adam | Reference paper |
| Learning rate | 0.001 | Implementation choice (paper does not state the exact value in the portion reviewed) |
| Loss | Binary cross-entropy (`BCELoss`) | Reference paper |
| Epochs | 30 | Reference paper |
| Batch size | 32 | Implementation choice |
| Decision threshold | 0.5 (segment) and >0.5 majority (recording) | Standard / reference paper |
| Cross-validation | 5-fold, person-independent | Reference paper / corpus |
| Repetitions (R) | **1** | Deviation — paper uses R = 5 |

### 5.4 Random seeds and reproducibility

**No random seed is set anywhere in the pipeline.** Model weights are randomly initialised, and the training batch order is shuffled each epoch, so **exact numerical results vary from run to run**. This is a known and expected source of variance (see Section 9) and directly explains why two executions of the same code produced slightly different fold-level numbers. Setting a fixed seed is recommended before final reporting.

### 5.5 Note on model output location

The notebook writes trained models to `../Models/Baseline_model_v2/` (creation cell) but loads the saved results summary from `../Models/Baseline_model/`. **[Information Missing – Please Complete: confirm which directory holds the definitive saved models and `results_summary.json`, and reconcile the two paths.]** The results reported in Section 8 are those loaded from `results_summary.json`.

---

## 6. Baseline Implementation

The pipeline proceeds in six stages. Stages 1–3 were completed prior to this report; Stages 4–6 constitute the classification baseline itself.

### 6.1 Stage 1 — Transcription

Whisper `large-v3` transcribes each `.wav` file with `language="it"` forced (rather than auto-detected) for robustness on short or noisy spontaneous speech. Output preserves the `HC`/`PT` folder structure and filename stems (`transcripts/HC/…`, `transcripts/PT/…`). A resume check skips already-transcribed files, allowing safe recovery from interrupted GPU sessions. Verified: 52 HC + 64 PT = 116 transcripts.

### 6.2 Stage 2 — Segmentation

Each transcript is split into N = 1, 2, 4, 8, 16, 32, 64 contiguous, near-equal chunks by word count. Files are named `<stem>_N{n}_seg{k}.txt` in flat `segments/HC/` and `segments/PT/` directories. The flat naming (rather than nested per-speaker folders) was chosen so that "all segments at a given N across all speakers" can be retrieved with a single glob pattern — matching the paper's design of training one classifier per N-level. Verified: 127 segments per transcript (1+2+4+8+16+32+64), 14,732 total, exact reconstruction confirmed.

### 6.3 Stage 3 — Feature extraction

Each segment is tokenised and passed through `dbmdz/bert-base-italian-xxl-cased`; the per-token 768-dimensional hidden states are **mean-pooled** into a single 768-dimensional vector per segment, saved as a `.npy` file mirroring the segment's name and folder.

**Key decisions:**

- **Model choice.** UmBERTo (`Musixmatch/umberto-commoncrawl-cased-v1`) was attempted first but failed to load due to a tokenizer-format incompatibility with the current `transformers` library. `dbmdz/bert-base-italian-xxl-cased` — an actively maintained Italian-native BERT with a standard WordPiece tokenizer — was used instead.
- **Rejection of LIWC alternatives.** Empath (English-only), the NRC Emotion Lexicon (machine-translated Italian, low-dimensional), and FEEL-IT (2–4 output dimensions, Twitter domain, no pronoun/cognitive-process signal) were each considered and rejected in favour of a general embedding representation.
- **Known truncation limit.** Inputs longer than 512 tokens are truncated. This mainly affects N = 1 (whole transcript) and some N = 2 segments; N ≥ 8 segments always fit.

### 6.4 Stage 4 — Data loading and fold assignment

Two supporting functions were built and verified before any model was trained:

- `speaker_to_fold`: parses `fold-lists.csv` (Interview-Task columns 7–11), stripping literal quote characters, to map each of the 116 speakers to a fold (1–5).
- `speaker_from_filename()`: uses a regular expression (`_N\d+_seg\d+$`) to recover the speaker stem from any feature filename, regardless of N-level or segment index.
- `load_data_for_n(n)`: loads every feature vector for a given N-level, together with its label (0 for HC, 1 for PT — taken from the source folder), its fold number, and its speaker id, returning four aligned NumPy arrays.

**Verification performed:** speaker sets from filenames and from the fold list match exactly (116 = 116); no speaker leakage across folds; array shapes and per-class counts match the expected `116 × N` structure.

### 6.5 Stage 4 — Model, training, and splitting

- **Model** (`DepressionClassifier`): `nn.Sequential` of `Linear(768,32) → ReLU → Linear(32,64) → ReLU → Linear(64,128) → ReLU → Linear(128,1) → Sigmoid`.
- **Training** (`train_model`): Adam (lr 0.001), `BCELoss`, 30 epochs, batch size 32, with per-epoch shuffling. A fresh, randomly initialised model is trained for every fold.
- **Splitting** (`get_fold_split`): boolean masking selects the held-out fold as the test set and all remaining folds as the training set, keeping `X`, `y`, and `speaker` arrays aligned.

### 6.6 Stage 5 — Aggregation to recording level

`evaluate_model` produces two levels of result:

- **Segment-level:** accuracy and F1 over individual segment predictions (threshold 0.5).
- **Recording-level:** for each speaker in the test fold, the fraction of that speaker's segments predicted "Depression" is computed; if it exceeds 0.5, the speaker is labelled Depression, else Control (majority vote). Accuracy and F1 are then computed over these per-speaker decisions. Recording-level figures are the ones comparable to the reference paper.

### 6.7 Stage 6 — Cross-validation driver

`run_cross_validation(n)` loads the data for one N-level, iterates over the five folds (training a fresh model and evaluating each time), and returns the mean and standard deviation of segment- and recording-level accuracy and F1. An outer loop applies this across all seven N-levels; each of the 35 trained models (7 N × 5 folds) is saved as a `.pt` file, and the aggregate results are serialised to `results_summary.json`.

### 6.8 Workflow diagram (textual)

```
For each N in {1,2,4,8,16,32,64}:
    load all feature vectors for that N  (X, y, fold, speaker)
    For each fold F in {1..5}:
        split -> train = folds != F, test = fold == F   (person-independent)
        train a fresh FFNN (Adam, BCE, 30 epochs)
        predict each test segment
        majority-vote per speaker -> recording-level decision
        record segment- and recording-level accuracy & F1
    average across the 5 folds  ->  mean ± std for this N
```

---

## 7. Evaluation

### 7.1 Metrics

- **Accuracy** — fraction of correct predictions. Reported for completeness.
- **F1 score** — harmonic mean of precision and recall; the primary metric, because it is more robust to the corpus's class imbalance (52 HC vs 64 PT) and is the metric emphasised by the reference paper.

Both metrics are computed at two levels: **segment-level** (per individual segment) and **recording-level** (per speaker, after majority vote). Recording-level is the metric used for benchmark comparison.

### 7.2 Evaluation procedure

For each N-level, the model is trained and evaluated five times (once per held-out fold). Each fold's recording-level accuracy and F1 are collected, and the mean and standard deviation across the five folds are reported. Because every speaker appears in the test fold exactly once across the five folds, every speaker is evaluated exactly once.

### 7.3 Observations during evaluation

- **Fast-converging training loss** (e.g. from ~1.8 to ~0.02 over 30 epochs) indicated a tendency toward overfitting, consistent with a 768-dimensional input against only 736–928 training examples per fold. This did not prevent generalisation but is reflected in the wider fold-to-fold variance.
- **Prediction distribution checks** confirmed the classifier was not collapsing to a single class (e.g. 89 vs 103 across a 96/96 test fold), so accuracy is meaningful and not an artefact of class imbalance.
- **Majority-vote aggregation improved results** substantially over segment-level scoring (e.g. at N = 8, segment-level ~83% accuracy vs recording-level ~89–91%), because averaging many segment votes per speaker smooths out isolated segment errors.

---

## 8. Results Summary

### 8.1 Baseline results (this work)

Recording-level results (after majority-vote aggregation), averaged across the five folds, one training run per fold. These are the definitive baseline numbers, loaded from `results_summary.json`.

| N (segments) | Accuracy | F1 score |
|---|---|---|
| 1 | 88.80% ± 5.87% | **90.27% ± 3.95%** |
| 2 | 87.97% ± 6.85% | 88.79% ± 5.59% |
| 4 | 88.80% ± 8.04% | 90.03% ± 5.80% |
| 8 | 88.88% ± 6.80% | 89.93% ± 5.84% |
| 16 | 86.23% ± 11.76% | 87.47% ± 9.60% |
| 32 | 81.96% ± 8.24% | 83.29% ± 6.65% |
| 64 | 80.22% ± 11.76% | 81.04% ± 9.89% |

*Table 8.1 — Baseline recording-level accuracy and F1 by segmentation level N (mean ± standard deviation across 5 folds, single run per fold).*

### 8.2 Comparison against reference benchmarks

| Benchmark | Accuracy | F1 score | Notes |
|---|---|---|---|
| Androids Corpus baseline (Tao et al., 2023) | 83.9% ± 1.3% | 84.7% ± 0.9% | Corpus's own reference baseline |
| ICMI 2025 — text-only, LIWC, SIL (best N) | — | 77.7% ± 1.0% | N = 4 |
| ICMI 2025 — text-only, LIWC, MIL (best N) | — | 76.2% ± 1.2% | N = 2 |
| ICMI 2025 — multimodal MIL (speech + text) | 92.5% ± 1.1% | 93.1% ± 1.1% | Paper's best overall |
| **This work — text-only embeddings (best N)** | **88.80% ± 5.87%** | **90.27% ± 3.95%** | N = 1, SIL, single run/fold |

*Table 8.2 — Baseline results in context. Sources: Tao et al. (2023); Alsarrani et al. (2025).*

### 8.3 Key findings

- The embedding-based text branch **exceeds the reference paper's best text-only (LIWC) result at every N-level tested**.
- Its best configuration (N = 1) **approaches the paper's best multimodal result**, using text alone.
- **Performance declines monotonically as N increases** (from ~90% F1 at N = 1 to ~81% F1 at N = 64) — the *opposite* trend to the paper's LIWC text results, which stay roughly flat until N = 32 (see Section 9.3 for interpretation).
- The three finest results (N = 1, 4, 8) lie within one standard deviation of each other and are best treated as **roughly tied**, rather than N = 1 being a decisive optimum.

---

## 9. Discussion

### 9.1 Strengths

- **Verified correctness end-to-end.** Every stage was checked (file counts, reconstruction, shape/NaN/zero checks, exact speaker–fold matching, explicit leakage test), giving high confidence that the results are not artefacts of a data-handling error.
- **Faithful to the reference architecture.** The classifier, optimiser, loss, epoch count, cross-validation protocol, and aggregation match the paper, isolating the feature representation as the main deliberate change.
- **Strong headline performance** relative to published text-only benchmarks.
- **Reproducible artefacts.** All 35 trained models and a machine-readable results summary are saved, so the numbers can be recovered without retraining.

### 9.2 Weaknesses and known limitations

- **Feature substitution.** Using BERT embeddings instead of LIWC means results are **not a like-for-like reproduction** of the paper's text branch; the comparison in Table 8.2 is against published numbers, not against a LIWC re-run on identical splits.
- **Single run per fold (R = 1).** The paper averages over R = 5 initialisations. This baseline's larger standard deviations (up to ±11.8%) partly reflect the absence of this averaging.
- **No fixed random seed**, so exact numbers are not bit-for-bit reproducible (Section 5.4).
- **512-token truncation** removes content from long low-N segments, a plausible contributor to the observed low-vs-high-N behaviour.
- **SIL only.** The MIL paradigm — central to the paper's punctual-vs-continuous question — is not implemented.
- **Loss of interpretability.** Unlike LIWC's psychologically named categories, BERT embedding dimensions are not directly interpretable, which constrains the kind of linguistic claims that can be made about *why* the model works.

### 9.3 Interpretation of the N-trend

The paper's LIWC text results degrade mainly at high N, attributed to word-count statistics becoming unstable when segments are very short. This baseline instead degrades steadily from low to high N. A plausible explanation is that **the two representations fail for opposite reasons**: LIWC needs enough words for stable category percentages (a high-N weakness), whereas mean-pooled BERT embeddings may lose information for long segments through 512-token truncation and, at very high N, from pooling very few tokens. The overall direction is consistent — extreme segmentation harms performance — but the mechanism differs. This is a substantive observation to raise with the supervisor.

### 9.4 Challenges encountered

- **GPU quota interruptions** during transcription (Colab), mitigated by resume-on-restart logic.
- **UmBERTo tokenizer incompatibility**, resolved by switching to `dbmdz/bert-base-italian-xxl-cased`.
- **LIWC access uncertainty**, which motivated the feature substitution and remains an open question for the supervisor.
- **Corpus redistribution restrictions**, which required excluding all data, transcripts, segments, and features from version control.

---

## 10. Foundation for Future Work

This baseline establishes the reference point and validated infrastructure on which the remainder of the dissertation will build. Specifically:

1. **RNN-based classifier.** The next planned experiment replaces the FFNN with a recurrent architecture (cf. Alsofyani & Vinciarelli, 2021), evaluated on the *same* features, folds, and metrics, so improvement can be measured directly against Table 8.1.
2. **MIL variant.** Implementing Multiple Instance Learning will allow this branch to address the paper's core punctual-vs-continuous question, comparing SIL against MIL under identical conditions.
3. **Repeated initialisation (R = 5).** Running each fold multiple times and averaging is expected to reduce the wide variance in Section 8 and bring the reporting protocol fully in line with the paper.
4. **LIWC comparison (pending access).** If institutional LIWC access is confirmed, re-running Stage 3 with LIWC would enable a genuine like-for-like comparison against the paper and quantify the embeddings-vs-LIWC difference.
5. **Reproducibility hardening.** Fixing a random seed and recording exact library versions will make final results reproducible.
6. **Eventual multimodal fusion.** Although out of scope for this branch, the recording-level outputs produced here are in the correct form to later combine with the speech branch.

All of the above will reuse the verified data-loading, fold-assignment, and evaluation code documented in Section 6, changing only the model or the feature representation — which is precisely what makes a documented baseline valuable.

---

## 11. References

1. **Alsarrani, R., Esposito, A., & Vinciarelli, A. (2025).** *Punctual or Continuous? Analyzing Depression Traces in Language and Paralanguage with Multiple Instance Learning.* Proceedings of the 27th ACM International Conference on Multimodal Interaction (ICMI 2025). — **Primary baseline reference.**
2. **Tao, F., Esposito, A., & Vinciarelli, A. (2023).** *The Androids Corpus: A New Publicly Available Benchmark for Speech Based Depression Detection.* Proceedings of Interspeech 2023. — Dataset and corpus baseline.
3. **Alsofyani, H., & Vinciarelli, A. (2021).** *[Stacked Recurrent Neural Networks — Interspeech 2021].* Secondary architectural reference for planned RNN work. **[Information Missing – Please Complete: exact title as cited in your bibliography.]**
4. **Whisper (OpenAI)** — `large-v3` automatic speech recognition model, used for transcription. **[Information Missing – Please Complete: preferred citation.]**
5. **`dbmdz/bert-base-italian-xxl-cased`** — Italian BERT model (Bavarian State Library / dbmdz), used for feature extraction. **[Information Missing – Please Complete: preferred citation.]**

---

*End of baseline reference document.*